# Lab Exercise: SQL Analysis with Polars

In this lab, you'll practice SQL queries using Polars' built-in SQL functionality. Complete each exercise by writing the appropriate SQL query.

In [65]:
# Setup - Run this cell first
import polars as pl

# Load data
airlines = pl.read_csv(
    'https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airlines.csv'
)

airports = pl.read_csv(
    'https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airports.csv'
)

flights = pl.read_csv(
    'https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_flights.csv',
    null_values='NA'
)

planes = pl.read_csv(
    'https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_planes.csv',
    null_values='NA'
)

# Weather CSV fix: treat airport codes as null, ignore errors, cast precip
weather = (
    pl.read_csv(
        "https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_weather.csv",
        null_values=["EWR", "JFK", "LGA"],
        truncate_ragged_lines=True,
        infer_schema_length=10000,
        ignore_errors=True
    )
    .with_columns(pl.col("precip").cast(pl.Float64, strict=False))
)

# Convert time_hour to datetime
flights = flights.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))
weather = weather.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))

# Create SQL context
ctx = pl.SQLContext(
    airlines=airlines,
    airports=airports,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True
)

print("Setup complete! Tables available:")
print(ctx.execute("SHOW TABLES"))


Setup complete! Tables available:
shape: (5, 1)
┌──────────┐
│ name     │
│ ---      │
│ str      │
╞══════════╡
│ airlines │
│ airports │
│ flights  │
│ planes   │
│ weather  │
└──────────┘


/tmp/ipython-input-2445374245.py:40: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


## Exercise 1: Basic Queries

### 1.1 Find all unique carriers in the airlines table

In [66]:
# Write your SQL query here
result = ctx.execute("""
SELECT DISTINCT carrier
FROM airlines
""")

print(result)

shape: (16, 1)
┌─────────┐
│ carrier │
│ ---     │
│ str     │
╞═════════╡
│ 9E      │
│ AA      │
│ AS      │
│ B6      │
│ DL      │
│ …       │
│ UA      │
│ US      │
│ VX      │
│ WN      │
│ YV      │
└─────────┘


### 1.2 Find the top 10 destinations by number of flights

In [67]:
# Write your SQL query here
result = ctx.execute("""
SELECT
    dest,
    COUNT(*) AS num_flights
FROM flights
GROUP BY dest
ORDER BY num_flights DESC
LIMIT 10
""")

print(result)

shape: (10, 2)
┌──────┬─────────────┐
│ dest ┆ num_flights │
│ ---  ┆ ---         │
│ str  ┆ u32         │
╞══════╪═════════════╡
│ ORD  ┆ 17283       │
│ ATL  ┆ 17215       │
│ LAX  ┆ 16174       │
│ BOS  ┆ 15508       │
│ MCO  ┆ 14082       │
│ CLT  ┆ 14064       │
│ SFO  ┆ 13331       │
│ FLL  ┆ 12055       │
│ MIA  ┆ 11728       │
│ DCA  ┆ 9705        │
└──────┴─────────────┘


### 1.3 Find all flights that departed more than 2 hours late (120 minutes)

In [68]:
# Write your SQL query here
result = ctx.execute("""
SELECT *
FROM flights
WHERE dep_delay > 120
""")

print(result)

shape: (9_723, 19)
┌──────┬───────┬─────┬──────────┬───┬──────────┬──────┬────────┬─────────────────────────┐
│ year ┆ month ┆ day ┆ dep_time ┆ … ┆ distance ┆ hour ┆ minute ┆ time_hour               │
│ ---  ┆ ---   ┆ --- ┆ ---      ┆   ┆ ---      ┆ ---  ┆ ---    ┆ ---                     │
│ i64  ┆ i64   ┆ i64 ┆ i64      ┆   ┆ i64      ┆ i64  ┆ i64    ┆ datetime[μs, UTC]       │
╞══════╪═══════╪═════╪══════════╪═══╪══════════╪══════╪════════╪═════════════════════════╡
│ 2013 ┆ 1     ┆ 1   ┆ 848      ┆ … ┆ 184      ┆ 18   ┆ 35     ┆ 2013-01-01 23:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 957      ┆ … ┆ 200      ┆ 7    ┆ 33     ┆ 2013-01-01 12:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 1114     ┆ … ┆ 1416     ┆ 9    ┆ 0      ┆ 2013-01-01 14:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 1540     ┆ … ┆ 1598     ┆ 13   ┆ 38     ┆ 2013-01-01 18:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 1815     ┆ … ┆ 1134     ┆ 13   ┆ 25     ┆ 2013-01-01 18:00:00 UTC │
│ …    ┆ …     ┆ …   ┆ …        ┆ … ┆ …        ┆ …    ┆ …      ┆ …     

## Exercise 2: Aggregation

### 2.1 Calculate the average departure delay for each origin airport

In [69]:
# Write your SQL query here
result = ctx.execute("""
SELECT
    origin,
    AVG(dep_delay) as avg_delay
FROM flights
WHERE dep_delay IS NOT NULL
GROUP BY origin
ORDER BY avg_delay DESC
""")

print(result)

shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘


### 2.2 Find the busiest month of the year

Count the number of flights per month and find which month has the most flights.

In [70]:
# First, let's check what columns are available
result = ctx.execute("""
    SELECT *
    FROM flights
    LIMIT 5
""")
# print(result)

# Now write your query to find busiest month
result = ctx.execute("""
SELECT
    month,
    COUNT(*) as num_flights
FROM flights
GROUP BY month
ORDER BY num_flights DESC
LIMIT 1
""")

print(result)

shape: (1, 2)
┌───────┬─────────────┐
│ month ┆ num_flights │
│ ---   ┆ ---         │
│ i64   ┆ u32         │
╞═══════╪═════════════╡
│ 7     ┆ 29425       │
└───────┴─────────────┘


### 2.3 Calculate the on-time performance rate for each carrier

Consider a flight on-time if the departure delay is <= 15 minutes.

In [71]:
# Write your SQL query here
result = ctx.execute("""
SELECT
    carrier,
    CAST(SUM(CASE WHEN dep_delay <= 15 THEN 1 ELSE 0 END) AS FLOAT) * 100 / COUNT(*) AS on_time_rate
FROM flights
WHERE dep_delay IS NOT NULL
GROUP BY carrier
ORDER BY on_time_rate DESC
""")

print(result)

shape: (16, 2)
┌─────────┬──────────────┐
│ carrier ┆ on_time_rate │
│ ---     ┆ ---          │
│ str     ┆ f64          │
╞═════════╪══════════════╡
│ HA      ┆ 92.982456    │
│ US      ┆ 87.822674    │
│ AS      ┆ 86.797753    │
│ AA      ┆ 84.071293    │
│ DL      ┆ 83.681246    │
│ …       ┆ …            │
│ FL      ┆ 73.32915     │
│ WN      ┆ 73.102706    │
│ F9      ┆ 71.847507    │
│ YV      ┆ 71.376147    │
│ EV      ┆ 69.538126    │
└─────────┴──────────────┘


## Exercise 3: Joins

### 3.1 List all flights with their airline names (not just carrier codes)

Show the first 20 flights with carrier code, airline name, flight number, origin, and destination.

In [72]:
# Write your SQL query here
result = ctx.execute("""
SELECT
    f.carrier,
    a.name AS airline_name,
    f.flight,
    f.origin,
    f.dest
FROM flights AS f
JOIN airlines AS a ON f.carrier = a.carrier
LIMIT 20
""")

print(result)

shape: (20, 5)
┌─────────┬────────────────────────┬────────┬────────┬──────┐
│ carrier ┆ airline_name           ┆ flight ┆ origin ┆ dest │
│ ---     ┆ ---                    ┆ ---    ┆ ---    ┆ ---  │
│ str     ┆ str                    ┆ i64    ┆ str    ┆ str  │
╞═════════╪════════════════════════╪════════╪════════╪══════╡
│ UA      ┆ United Air Lines Inc.  ┆ 1545   ┆ EWR    ┆ IAH  │
│ UA      ┆ United Air Lines Inc.  ┆ 1714   ┆ LGA    ┆ IAH  │
│ AA      ┆ American Airlines Inc. ┆ 1141   ┆ JFK    ┆ MIA  │
│ B6      ┆ JetBlue Airways        ┆ 725    ┆ JFK    ┆ BQN  │
│ DL      ┆ Delta Air Lines Inc.   ┆ 461    ┆ LGA    ┆ ATL  │
│ …       ┆ …                      ┆ …      ┆ …      ┆ …    │
│ B6      ┆ JetBlue Airways        ┆ 1806   ┆ JFK    ┆ BOS  │
│ UA      ┆ United Air Lines Inc.  ┆ 1187   ┆ EWR    ┆ LAS  │
│ B6      ┆ JetBlue Airways        ┆ 371    ┆ LGA    ┆ FLL  │
│ MQ      ┆ Envoy Air              ┆ 4650   ┆ LGA    ┆ ATL  │
│ B6      ┆ JetBlue Airways        ┆ 343    ┆ EWR    ┆ 

### 3.2 Find the average age of planes for each carrier

Hint: The planes table has a `year` column for manufacture year. Calculate age based on 2013.

In [73]:
# Write your SQL query here
result = ctx.execute("""
SELECT
    f.carrier,
    AVG(2013 - p.year) AS average_plane_age
FROM flights AS f
JOIN planes AS p ON f.tailnum = p.tailnum
WHERE p.year IS NOT NULL
GROUP BY f.carrier
ORDER BY average_plane_age DESC
""")

print(result)

shape: (16, 2)
┌─────────┬───────────────────┐
│ carrier ┆ average_plane_age │
│ ---     ┆ ---               │
│ str     ┆ f64               │
╞═════════╪═══════════════════╡
│ MQ      ┆ 35.319            │
│ AA      ┆ 25.869426         │
│ DL      ┆ 16.372169         │
│ UA      ┆ 13.207691         │
│ FL      ┆ 11.385829         │
│ …       ┆ …                 │
│ B6      ┆ 6.686702          │
│ F9      ┆ 4.87874           │
│ VX      ┆ 4.473643          │
│ AS      ┆ 3.33662           │
│ HA      ┆ 1.548387          │
└─────────┴───────────────────┘


### 3.3 Find flights that experienced both departure delays and bad weather

Join flights with weather data and find flights where departure delay > 30 minutes and either wind_speed > 20 or precip > 0.1

In [74]:
# Ensure numeric weather columns
weather = weather.with_columns([
    pl.col("wind_speed").cast(pl.Float64, strict=False),
    pl.col("precip").cast(pl.Float64, strict=False)
])

# Re-create SQL context with updated weather
ctx = pl.SQLContext(
    airlines=airlines,
    airports=airports,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True
)

# Query: flights with dep_delay > 30 and bad weather
query = """
SELECT f.*, w.wind_speed, w.precip
FROM flights f
JOIN weather w
  ON f.origin = w.origin AND f.time_hour = w.time_hour
WHERE f.dep_delay > 30
  AND (w.wind_speed > 20 OR w.precip > 0.1)
"""

result = ctx.execute(query)
result


/tmp/ipython-input-4201425508.py:8: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,wind_speed,precip
i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,i64,i64,i64,i64,"datetime[μs, UTC]",f64,f64


## Exercise 4: Advanced Queries

### 4.1 Find the most popular aircraft types (by number of flights)

Join flights with planes to get manufacturer and model information. Show top 10.

In [75]:
# Write your SQL query here
result = ctx.execute
query = """
SELECT p.manufacturer, p.model, COUNT(*) AS num_flights
FROM flights f
JOIN planes p
  ON f.tailnum = p.tailnum
GROUP BY p.manufacturer, p.model
ORDER BY num_flights DESC
LIMIT 10
"""

result = ctx.execute(query)
result


# print(result)

manufacturer,model,num_flights
str,str,u32
"""AIRBUS""","""A320-232""",31278
"""EMBRAER""","""EMB-145LR""",28027
"""EMBRAER""","""ERJ 190-100 IGW""",23716
"""AIRBUS INDUSTRIE""","""A320-232""",14553
"""EMBRAER""","""EMB-145XR""",14051
"""BOEING""","""737-824""",13809
"""BOMBARDIER INC""","""CL-600-2D24""",11807
"""BOEING""","""737-7H4""",10389
"""BOEING""","""757-222""",9150


### 4.2 Analyze route performance

Find the top 10 routes (origin-destination pairs) with:
- Total number of flights
- Average departure delay
- Percentage of flights delayed more than 30 minutes

Include airport names, not just codes.

In [76]:
# Write your SQL query here
# Rename airport name columns to avoid conflicts
airports_origin = airports.select([
    pl.col("faa").alias("origin"),
    pl.col("name").alias("origin_name")
])

airports_dest = airports.select([
    pl.col("faa").alias("dest"),
    pl.col("name").alias("dest_name")
])

# Join flights with renamed airports
query = """
SELECT
    f.origin,
    ao.origin_name,
    f.dest,
    ad.dest_name,
    COUNT(*) AS total_flights,
    AVG(f.dep_delay) AS avg_dep_delay,
    100.0 * SUM(CASE WHEN f.dep_delay > 30 THEN 1 ELSE 0 END) / COUNT(*) AS pct_delayed_over_30
FROM flights f
JOIN airports_origin ao ON f.origin = ao.origin
JOIN airports_dest ad ON f.dest = ad.dest
GROUP BY f.origin, ao.origin_name, f.dest, ad.dest_name
ORDER BY total_flights DESC
LIMIT 10
"""

# Create a new SQLContext with the renamed airport tables
ctx = pl.SQLContext(
    airlines=airlines,
    airports_origin=airports_origin,
    airports_dest=airports_dest,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True
)

result = ctx.execute(query)
result


/tmp/ipython-input-1201500744.py:32: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


origin,origin_name,dest,dest_name,total_flights,avg_dep_delay,pct_delayed_over_30
str,str,str,str,u32,f64,f64
"""JFK""","""John F Kennedy Intl""","""LAX""","""Los Angeles Intl""",11262,8.522508,9.829515
"""LGA""","""La Guardia""","""ATL""","""Hartsfield Jackson Atlanta Int…",10263,11.448621,12.247881
"""LGA""","""La Guardia""","""ORD""","""Chicago Ohare Intl""",8857,10.740758,13.345377
"""JFK""","""John F Kennedy Intl""","""SFO""","""San Francisco Intl""",8204,11.952691,12.116041
"""LGA""","""La Guardia""","""CLT""","""Charlotte Douglas Intl""",6168,8.965321,11.948768
"""EWR""","""Newark Liberty Intl""","""ORD""","""Chicago Ohare Intl""",6100,14.644163,16.0
"""JFK""","""John F Kennedy Intl""","""BOS""","""General Edward Lawrence Logan …",5898,11.694953,13.767379
"""LGA""","""La Guardia""","""MIA""","""Miami Intl""",5781,7.361747,9.462031
"""JFK""","""John F Kennedy Intl""","""MCO""","""Orlando Intl""",5464,10.601583,12.719619


## Bonus: Compare with Polars

### Choose one of the queries above and implement it using Polars

This will help you understand the relationship between SQL and Polars operations.

In [77]:
# Example: Let's implement Exercise 2.1 (average delay by origin) in Polars

# SQL version (for reference)
sql_result = ctx.execute("""
    SELECT
        origin,
        AVG(dep_delay) as avg_delay
    FROM flights
    WHERE dep_delay IS NOT NULL
    GROUP BY origin
    ORDER BY avg_delay DESC
""")

# Polars version
polars_result = (
    flights
    .filter(pl.col('dep_delay').is_not_null())
    .group_by('origin')
    .agg(pl.col('dep_delay').mean().alias('avg_delay'))
    .sort('avg_delay', descending=True)
)

print("SQL Result:")
print(sql_result)
print("\nPolars Result:")
print(polars_result)

# Now implement one of your own queries in Polars below:
# Your Polars code here


SQL Result:
shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘

Polars Result:
shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘


In [78]:
import polars as pl

# Make sure flights and airports are Polars DataFrames
print(type(flights), type(airports))  # should both be <class 'polars.internals.frame.DataFrame'>

# Rename airport columns
airports_origin = airports.select([
    pl.col("faa").alias("origin"),
    pl.col("name").alias("origin_name")
])

airports_dest = airports.select([
    pl.col("faa").alias("dest"),
    pl.col("name").alias("dest_name")
])

# Join flights with airports (eager)
routes = flights.join(airports_origin, on="origin", how="left").join(airports_dest, on="dest", how="left")

# Confirm routes is a Polars DataFrame
print(type(routes))

# Group by origin/destination and aggregate
route_stats = routes.group_by(["origin", "origin_name", "dest", "dest_name"]).agg([
    pl.count().alias("total_flights"), # Use pl.count() or pl.col("any_column").count()
    pl.col("dep_delay").mean().alias("avg_dep_delay"),
    (100 * (pl.col("dep_delay") > 30).sum() / pl.count()).alias("pct_delayed_over_30") # Use pl.count()
])

# Sort and take top 10
route_stats = route_stats.sort("total_flights", descending=True).head(10)

route_stats

<class 'polars.dataframe.frame.DataFrame'> <class 'polars.dataframe.frame.DataFrame'>
<class 'polars.dataframe.frame.DataFrame'>


/tmp/ipython-input-1812372412.py:25: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("total_flights"), # Use pl.count() or pl.col("any_column").count()
/tmp/ipython-input-1812372412.py:27: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  (100 * (pl.col("dep_delay") > 30).sum() / pl.count()).alias("pct_delayed_over_30") # Use pl.count()


origin,origin_name,dest,dest_name,total_flights,avg_dep_delay,pct_delayed_over_30
str,str,str,str,u32,f64,f64
"""JFK""","""John F Kennedy Intl""","""LAX""","""Los Angeles Intl""",11262,8.522508,9.829515
"""LGA""","""La Guardia""","""ATL""","""Hartsfield Jackson Atlanta Int…",10263,11.448621,12.247881
"""LGA""","""La Guardia""","""ORD""","""Chicago Ohare Intl""",8857,10.740758,13.345377
"""JFK""","""John F Kennedy Intl""","""SFO""","""San Francisco Intl""",8204,11.952691,12.116041
"""LGA""","""La Guardia""","""CLT""","""Charlotte Douglas Intl""",6168,8.965321,11.948768
"""EWR""","""Newark Liberty Intl""","""ORD""","""Chicago Ohare Intl""",6100,14.644163,16.0
"""JFK""","""John F Kennedy Intl""","""BOS""","""General Edward Lawrence Logan …",5898,11.694953,13.767379
"""LGA""","""La Guardia""","""MIA""","""Miami Intl""",5781,7.361747,9.462031
"""JFK""","""John F Kennedy Intl""","""MCO""","""Orlando Intl""",5464,10.601583,12.719619
